# DIT5411 — Traditional Chinese Handwriting Recognition (Kaggle)

**Settings → Accelerator → GPU**, **Internet → On**.

Full config: 128px, 1 epoch of 20,394 steps (~2.5h per model) — this reaches real accuracy. Train one model per cell, Quick Save between models. Paste `RESULTS.md` to me for the GitHub repo.

In [ ]:
import os, sys, subprocess, glob

if not os.path.isfile('train.py'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/cveg1234/DIT5411-LuiYuHin.git'], check=True)
    os.chdir('DIT5411-LuiYuHin')
subprocess.run(['git', 'pull', '-q'])
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('Working dir:', os.getcwd())
print('GPU devices:', gpus)
if not gpus:
    print('WARNING: no GPU detected. Set Settings -> Accelerator -> GPU T4 before the full run.')
print('Setup done.')

## Download & extract the dataset (one time, ~15 min)
684,677 images across 13,065 characters. If `data/cleaned_data` already exists (e.g. from a saved Output), this cell skips.

In [ ]:
DATA_DIR = os.path.join(os.getcwd(), 'data', 'cleaned_data')
if not os.path.isdir(DATA_DIR) or len(os.listdir(DATA_DIR)) < 1000:
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/chenkenanalytic/handwritting_data_all.git'], check=True)
    with open('all_data.zip', 'wb') as out:
        for part in sorted(glob.glob('handwritting_data_all/all_data.zip.*')):
            with open(part, 'rb') as f:
                out.write(f.read())
    subprocess.run([sys.executable, 'extract_dataset.py',
                    '--zip', 'all_data.zip', '--dest', DATA_DIR], check=True)
print('Character folders:', len(os.listdir(DATA_DIR)))

## Quick pilot (optional — 20 classes, 2 epochs)

In [ ]:
r = subprocess.run([sys.executable, 'train.py', '--data-dir', DATA_DIR,
                    '--output-dir', 'results_pilot', '--num-classes', '20',
                    '--epochs', '2', '--models', 'A', 'B', '--no-imagenet',
                    '--steps-per-epoch', '30'])
print('pilot exit code:', r.returncode)
print(open('results_pilot/RESULTS.md', encoding='utf-8').read())

## (Optional) Continue training from your backed-up models
To train **one more epoch** starting from the models you downloaded:
1. In Kaggle's **Output/Files sidebar** (`/kaggle/working`), click the **upload** button and upload `model_A.keras` and `model_B.keras` into the project folder (next to `train.py`).
2. Run the cell below — it copies them into the checkpoint folder and marks them as 1 completed epoch.
3. Then run the model cells with `--epochs 2 --resume` (the state says 1 done, so only epoch 2 trains).


In [ ]:
import glob, os, shutil, json
# Check what was uploaded
print('Uploaded .keras files:', glob.glob('*.keras') + glob.glob('uploads/*.keras'))
subprocess.run([sys.executable, 'prepare_resume.py'])
print('Checkpoints now:', glob.glob('results/checkpoints/*_state.json'))

## Train models — 1 full epoch each (~2.5h)
Full 20,394 steps/epoch at 128px. This config produced ~22% accuracy in the earlier full run. **1 epoch** is enough — no step cap.

Run **Model A**, then **Quick Save**, then **Model B** in the saved version (Copy & Run). Each `--resume` continues from checkpoints. Two models fit in ~5h and give a valid comparison. Model C can wait.

**No GitHub upload** — paste `RESULTS.md` to me when done.

In [ ]:
# ---- Model A: continue ONE more epoch from your saved 71.99% checkpoint ----
# Requires: setup cell, dataset cell, and the prepare_resume.py cell (state = 1 epoch done).
print('Continuing Model A from saved checkpoint (epoch 2 only)...', flush=True)
r = subprocess.run([sys.executable, 'train.py', '--data-dir', DATA_DIR,
                    '--output-dir', 'results', '--epochs', '2',
                    '--models', 'A', '--resume'])
print('Model A exit code:', r.returncode)
print(open('results/RESULTS.md', encoding='utf-8').read() if os.path.isfile('results/RESULTS.md') else 'no results')

### Before training Model B: clear any stale checkpoint
Run this once. It removes only Model B's old checkpoint so training starts fresh (keeps Model A's result intact).

In [ ]:
# ---- Model B: continue ONE more epoch from your saved checkpoint ----
# WARNING: your saved Model B is the collapsed run (loss 42.2, 0.05%).
# Requires: setup, dataset, and prepare_resume.py cell (state = 1 epoch done).
print('Continuing Model B from saved checkpoint (epoch 2 only)...', flush=True)
r = subprocess.run([sys.executable, 'train.py', '--data-dir', DATA_DIR,
                    '--output-dir', 'results', '--epochs', '2',
                    '--models', 'B', '--resume'])
print('Model B exit code:', r.returncode)
print(open('results/RESULTS.md', encoding='utf-8').read() if os.path.isfile('results/RESULTS.md') else 'no results')

In [ ]:
# ---- Model C (short epochs, resumable) ----
print('Starting Model C training...', flush=True)
r = subprocess.run([sys.executable, 'train.py', '--data-dir', DATA_DIR,
                    '--output-dir', 'results', '--epochs', '20',
                    '--steps-per-epoch', '1000', '--lr', '0.0003',
                    '--models', 'C', '--resume'])
print('Model C exit code:', r.returncode)

## Final results (after A, B and C are done)

In [ ]:
from IPython.display import Markdown, display
p = 'results/RESULTS.md'
display(Markdown(open(p, encoding='utf-8').read()) if os.path.isfile(p)
        else 'RESULTS.md not generated yet — finish all models first.')

## Quick accuracy test for Models B and C (~5 min each)
These run a **small 2,000-class subset** with 300 steps to check whether each model learns at all — before spending 2.5h on a full run.

- **loss near ln(2000)=7.6 and flat** → not learning
- **loss dropping well below 7.6** → model is learning; worth a full run

Writes to `results_test_b` / `results_test_c` so your real results stay untouched.

In [ ]:
# ---- Model B (deep CNN) quick test ----
print('Testing Model B (deep CNN)...', flush=True)
r = subprocess.run([sys.executable, 'train.py', '--data-dir', DATA_DIR,
                    '--output-dir', 'results_test_b', '--num-classes', '2000',
                    '--epochs', '1', '--steps-per-epoch', '300',
                    '--models', 'B', '--lr', '0.0003', '--clipnorm'])
print('B test exit code:', r.returncode)
print(open('results_test_b/RESULTS.md', encoding='utf-8').read() if os.path.isfile('results_test_b/RESULTS.md') else 'no results')

In [ ]:
# ---- Model C (MLP baseline) quick test ----
print('Testing Model C (MLP)...', flush=True)
r = subprocess.run([sys.executable, 'train.py', '--data-dir', DATA_DIR,
                    '--output-dir', 'results_test_c', '--num-classes', '2000',
                    '--epochs', '1', '--steps-per-epoch', '300',
                    '--models', 'C', '--lr', '0.001'])
print('C test exit code:', r.returncode)
print(open('results_test_c/RESULTS.md', encoding='utf-8').read() if os.path.isfile('results_test_c/RESULTS.md') else 'no results')